# Climate for each basin (historical + future)

In [ ]:
from tqdm import tqdm
import pandas as pd
import os

import xarray as xr
import geopandas as gpd

from andeangc import config as cfg
from andeangc.polygon_extract import extract_timeseries

In [ ]:
basin_shp = gpd.read_file(cfg.VERSION / 'AndeanGC_shape.gpkg')[["gauge_id", "geometry"]]

## Historical climate

In [ ]:
# Historical data (1960-2024)
variables = ['prcp', 'tasmax', 'tasmin', 'tas', 'ep']

for variable in tqdm(variables):

    # Load only the yearly subset using time selection
    # Filename pattern comes from config.yml:inputs — the archive is stamped
    # with the span it was downloaded with, not with period_climate.
    file_path = cfg.data_dir('era5') / cfg.inputs['era5'].format(variable=variable)

    with xr.open_dataset(file_path) as ds:
        stack = (
            ds[variable]
            .sel(longitude=slice(
                    basin_shp.total_bounds[0],
                    basin_shp.total_bounds[2]),
                 latitude=slice(
                    basin_shp.total_bounds[1],
                    basin_shp.total_bounds[3]),
            )
        )
        
        # Extract timeseries for this period
        data = extract_timeseries(stack, basin_shp)
        data.to_parquet(cfg.VERSION / f'climate/historical/AndeanGC_{variable}_ERA5_1960_2025.parquet', compression= "zstd")

## Future climate

In [ ]:
# Future data (2025-2099)
variables = ['prcp', 'tasmax', 'tasmin']
gcms = list(cfg.data_dir('cmip6').glob('*'))

for gcm_path in tqdm(gcms, leave=True):
    gcm_name = os.path.basename(gcm_path).replace('.nc', '')

    for variable in tqdm(variables, position=1, leave=False):

        # Load only the 5-year subset using time selection
        if variable == 'prcp':
            var_name = 'pr'
        else:
            var_name = variable.lower()

        with xr.open_dataset(gcm_path)[var_name] as ds:
            stack = ds.sel(
                    longitude=slice(basin_shp.total_bounds[0], basin_shp.total_bounds[2]),
                    latitude=slice(basin_shp.total_bounds[1], basin_shp.total_bounds[3]))

            # Extract timeseries for this period
            data = extract_timeseries(stack, basin_shp)
            data.index = pd.to_datetime(data.index.astype(str)).normalize()
            data = data.asfreq('D')
            data.to_parquet(cfg.VERSION / f'climate/future/{variable}_{gcm_name}.parquet')